# Element Creation Examples

In this notebook, we will show some examples for how to create elements as trees.
We will try to start with the thought process, and walk you through the code as we create the desired model.

In [ ]:
import math
import numpy as np

from copy import deepcopy
from functools import partial
from pathlib import PurePath
from isotopes import Al, Si, Cd, He4, Cr, Cu, Fe, Mg, Mn, Ti, Zn, B, U234, U235, U236, U238
from itertools import chain

from coremaker.elements.box import BoxTree, ExcludeFrame, SplitBox, FrameBox, excludeframe_to_framebox
from coremaker.elements.assembly import singular_root_construction
from coremaker.elements.cylindrically_symmetric import AnnulusTree, ChunkedAnnulusTree, ChunkedCylinderTree, CylinderTree
from coremaker.materials.mixture import Mixture
from coremaker.materials.aluminium import al1050
from coremaker.materials.zirconium import zircalloy_4
from coremaker.materials.absorbers import hafnium, aic, b4c
from coremaker.materials.water import make_light_water
from coremaker.geometries import infiniteGeometry, FiniteCylinder, BareGeometry, Box as BoxGeometry, TriPrism
from coremaker.surfaces import Plane
from coremaker.transform import Transform
from coremaker.tree import Tree, Node, ChildType
import matplotlib.pyplot as plt

# Case 1: MTR-style fuel

We start with the [specification of the OPAL reactor fuel](https://apo.ansto.gov.au/server/api/core/bitstreams/cbb082fd-3e09-44e5-b6e1-ea1e0a122a8f/content), found on pages 3-4.
We show just the figure from those pages, to remind readers what a fuel plate looks like in general.

![OPAL fuel](OPALfuel.png)

The full specification can be a bit tedious to read, but readers who care to do so may open that link and look at the tables while reading this example.

## Breaking the object down
Our first order of business is to deconstruct the object into its underlying components.
In this process, we want to say at each stage if things sit next to each other, or within one another. These correspond to inclusive and exclusive relations between the progeny nodes and their parent node. These concepts are covered in detail in the [Elements](elements.rst) section of the reference guide.

For our MTR fuel assembly, we start with the outmost shape, which is a 81.5mmx81.5mmx1045mm virtual box, or we could imagine it to be within an infinite pool of water instead.
We notice that we can split it into two regions, the `fuel plate` region and the so-called `end box` region.
These sit one atop the other, where the `end box` region is 145mm tall.
In a real setting, outside the end box we would have some free room between this leg and the grid, followed by the core grid material on top of which the fuel rods sit.
For our simplified model, we will assume that outside the `end box` we have light water, and in the `fuel plate` region anything outside the defined geometry is filled with light water at room temperature as well.

We can now break down each of these, and once one of them reaches its simplest shape, we will model it.

## The End Box
The end box is made up of a corner-cut 69mmx69mm square made up of aluminium-6061, where the corners are cut at 8mm isosceles right angle triangles, set **within** our end box region full of water.
**Within** this corner-cut square, we have a cylinder full of water with radius 60mm that is concentric with the square.

Thus, our `end box` tree would be:
    
```mermaid
flowchart TD

A[Infinite Water] -->|External| B[Cut Square]
B -->|External| C[Water Cylinder]
```

We could build that as is, which has the benefit that it follows the actual shape of the components in the model. However, this introduces some complication, since we need to model the cut corners. These make that box region more complicated and has more surfaces, which can sometimes degrade performance on some codes.
Therefore, we are faced with a choice. We can have this simple tree model where the middle node has a complex geometry, or we can build a more complex tree with simpler geometries. We could define this tree instead:

```mermaid
flowchart TD

A[Infinite Water] -->|External| B[69mm Square]
B -->|External| C1[4x 8mmx8mm Isosceles right angle triangles]
B -->|External| C2[Water Cylinder]
```

This construction has more components which will lead to more Cells in an OpenMC-adapted model, but each of which has less surfaces.
It also uses more established geometry types on our end, which increases how idiomatic the model is.

Since we are trying to teach the reader, we will show both options.

### Option 1: Smaller tree, complex geometry
Let's start by creating the materials we use.
We will use the tool for light water that we directly support.

In [ ]:
water = make_light_water(25.)

Aluminium is given in table 7, we'll use the side plate specification for the end box material as well.

In [ ]:
al6061clad = Mixture.by_weight_fraction(
    {Al: 97.51e-2,
     Cr: 7e-4,
     Cu: 22e-4,
     Fe: 58e-4,
     Mg: 85e-4,
     Mn: 4e-4,
     Si: 63e-4,
     Ti: 5e-4,
     Zn: 4e-4,
     B: 12e-6, # EBC
    },
    density = 2.71, #g/cc
    temperature=20.,
)

al6061side = Mixture.by_weight_fraction(
    {Al: 97.36e-2,
     Cr: 11e-4,
     Cu: 21e-4,
     Fe: 70e-4,
     Mg: 86e-4,
     Mn: 2e-4,
     Si: 60e-4,
     Ti: 7e-4,
     Zn: 6e-4,
     B: 9e-6, # EBC
    },
    density = 2.71, #g/cc
    temperature=20.,
)

al6061 = al6061side  # Any use for Al6061 other than clad will use this alias.

Next, we need to make the infinite geometry and the inner cylinder.
The infiniteGeometry was already imported above, and since it is a singleton instance we don't need to do anything for it. Just the cylinder, then.
It is always best to create geometries with origin centers and leave any
shifts to the transformations in the tree, so we will take care of the z-shift of the end box with transformations.
Our system uses cm as the common unit of measurement, so the engineering-focused use of mm has to be converted.

In [ ]:
endbox_cyl = FiniteCylinder(
    center=(0, 0, 0),  # Always prefer origin centers and leave translations to the transforms
    radius=60e-1, #60 mm in cm
    length=145e-1,
    axis=(0, 0, 1), # Z-axis
)

We can now make these into the simpler Nodes.

In [ ]:
CELL_SIDE = 80.5e-1
endbox_leg_flow = Node(geometry=endbox_cyl, mixture=water)
pool_node = Node(geometry=BoxGeometry(center=(0, 0, 0), dimensions=(CELL_SIDE, CELL_SIDE, 145e-1)), mixture=water)

We can now build the complex geometry and its node

In [ ]:
endbox_full = BoxGeometry(center=(0, 0, 0), dimensions=(69e-1, 69e-1, 145e-1))

From the figure above we can tell the sides of the corners and of the aluminium square, so we can find a formula for the distance between the corner and the center.

In [ ]:
square_side = 69e-1  # cm
corner_side = 8e-1  # cm
z_height = 145e-1
b = corner_side*math.sqrt(2)
a = 1
c = (corner_side ** 2 - square_side ** 2) / 2
determinant = b ** 2 - 4*a*c
d = math.sqrt(2) * (-b + math.sqrt(determinant)) / (2*a)
triangle_volume = z_height * b ** 2 / 2
corners = [Plane(1, -1, 0, -d), Plane(1, 1, 0, -d), -Plane(1, -1, 0, d), -Plane(1, 1, 0, d)]
corners

To make sure we got this right, we can plot these lines.

In [ ]:
plt.figure()
bl = np.array([-square_side/2 + corner_side, -square_side/2])
br = np.array([square_side/2 - corner_side, -square_side/2])
tl = np.array([-square_side/2 + corner_side, square_side/2])
tr = np.array([square_side/2 - corner_side, square_side/2])
lb = np.array([-square_side/2, -square_side/2 + corner_side])
lt = np.array([-square_side/2, square_side/2 - corner_side])
rb = np.array([square_side/2, -square_side/2 + corner_side])
rt = np.array([square_side/2, square_side/2 - corner_side])

def p1(x): return x + d
def p2(x): return -x - d
def p3(x): return x - d
def p4(x): return -x + d

for v1, v2 in [(bl, br), (tl, tr), (lb, lt), (rb, rt)]:
    x, y = zip(v1, v2)
    plt.plot(x, y, 'k')

for v1, v2, f, c in [(tr, rt, p4, 'b'), (lt, tl, p1, 'r'), (lb, bl, p2, 'm'), (br, rb, p3, 'g')]:
    x = np.array([v1[0], v2[0]])
    y = f(x)
    plt.plot(x, y, c)

Which seems fine. So we can use the surfaces from the full box and from these corners to get a corner-cut box.

In [ ]:
endbox_cut_corners = BareGeometry(list(endbox_full.surfaces) + corners, _volume=endbox_full.volume - 4 * triangle_volume)
endbox_box_node = Node(endbox_cut_corners, mixture=al6061)

Now we can build the Tree

In [ ]:
end_box = Tree()
end_box.nodes[PurePath("endbox")] = pool_node
end_box.nodes[PurePath("endbox/box")] = endbox_box_node
end_box.nodes[PurePath("endbox/box/waterleg")] = endbox_leg_flow
end_box.exclusive[PurePath("endbox")] = [(PurePath("endbox/box"), endbox_box_node)]
end_box.exclusive[PurePath("endbox/box")] = [(PurePath("endbox/box/waterleg"), endbox_leg_flow)]
end_box.plot(size=(5,2), label_offset=2e-2)

### Option 2: Larger tree, simple geometry

There is a correct way to do this that takes slightly more work (we need to recompute some `Plane`s) and an incorrect way to do them that will likely bite you at some point, that takes less work and seems fine until it doesn't.

We'll start with the wrong way, and explain why it is wrong even though it succeeds.

#### The WRONG way to do things
<div class="alert alert-block alert-danger"><b>Warning:</b> This is not how you should do things, but shows you how you could mistakenly think to do them. Skip to the next subsection if you want to just see the right way to do things</div>
The only real change is that we use the full box geometry for the box node, and that we need the triangle nodes.

In [ ]:
tr_tri = TriPrism((-corners[-1], -Plane(1, 0, 0, square_side/2), -Plane(0, 1, 0, square_side/2)), z_height, (square_side/2 - corner_side/2, square_side/2 - corner_side/2, 0))
tl_tri = TriPrism((-corners[0], Plane(1, 0, 0, -square_side/2), -Plane(0, 1, 0, square_side/2)), z_height, (-square_side/2 + corner_side/2, square_side/2 - corner_side/2, 0))
bl_tri = TriPrism((-corners[1], Plane(1, 0, 0, -square_side/2), Plane(0, 1, 0, -square_side/2)), z_height, (-square_side/2 + corner_side/2, -square_side/2 + corner_side/2, 0))
br_tri = TriPrism((-corners[2], -Plane(1, 0, 0, square_side/2), Plane(0, 1, 0, -square_side/2)), z_height, (square_side/2 - corner_side/2, -square_side/2 + corner_side/2, 0))
triangles = [tr_tri, tl_tri, bl_tri, br_tri]
triangle_nodes = [Node(tri, mixture=water) for tri in triangles]

endbox_full_node = Node(endbox_full, mixture=al6061)

And now we can assemble the Tree:

In [ ]:
end_box_2 = Tree()
end_box_2.nodes[PurePath("endbox")] = pool_node
end_box_2.nodes[PurePath("endbox/box")] = endbox_full_node
end_box_2.exclusive[PurePath("endbox")] = [(PurePath("endbox/box"), endbox_full_node)]
for i, triangle in enumerate(triangle_nodes):
    end_box_2.nodes[PurePath(f"endbox/box/tri_{i}")] = triangle
end_box_2.exclusive[PurePath("endbox/box")] = [(PurePath(f"endbox/box/tri_{i}"), triangle) for i, triangle in enumerate(triangle_nodes)]
end_box_2.nodes[PurePath("endbox/box/waterleg")] = endbox_leg_flow
end_box_2.exclusive[PurePath("endbox/box")].extend([(PurePath("endbox/box/waterleg"), endbox_leg_flow)])
end_box_2.plot(label_offset=0.15)

If we don't have any computation errors, the planes should match our corners from before.

In [ ]:
for _, c in end_box_2.named_components():
    surfs = c.geometry.surfaces
    planes = [s for s in surfs if isinstance(s, Plane) and s.a[0] != 0 and s.a[1] !=0]
    assert set(planes).issubset(set(corners) | set([-s for s in corners])), set(planes) - set(corners)

#### The RIGHT way to do this
<div style="background-color: #d4edda; color: #155724; padding: 15px; border: 1px solid #c3e6cb; border-radius: 5px;"><b>Note:</b> This is how you should do things. Don't do the other one.</div>
We need to create the `TriPrism`s such that they are centered, and put the translation on the transform in the nodes, instead.

In [ ]:
side_offset = corner_side / 4  # You can do the math yourself, but I did :)
d = corner_side / 2  # You can do the math yourself, but I did :)
tr_tri = TriPrism((Plane(1, 1, 0, -d), -Plane(1, 0, 0, side_offset), -Plane(0, 1, 0, side_offset)), z_height, (0, 0, 0))
tl_tri = TriPrism((Plane(-1, 1, 0, -d), Plane(1, 0, 0, -side_offset), -Plane(0, 1, 0, side_offset)), z_height, (0, 0, 0))
bl_tri = TriPrism((Plane(-1, -1, 0, -d), Plane(1, 0, 0, -side_offset), Plane(0, 1, 0, -side_offset)), z_height, (0, 0, 0))
br_tri = TriPrism((Plane(1, -1, 0, -d), -Plane(1, 0, 0, side_offset), Plane(0, 1, 0, -side_offset)), z_height, (0, 0, 0))
triangles = [tr_tri, tl_tri, bl_tri, br_tri]
triangles

The Planes seem to be in the right direction. We can now do the transforms.

In [ ]:
off_center = square_side/2 - side_offset
transforms = [Transform(translation=(off_center, off_center, 0)),
              Transform(translation=(-off_center, off_center, 0)),
              Transform(translation=(-off_center, -off_center, 0)),
              Transform(translation=(off_center, -off_center, 0))
             ]
triangle_nodes = [Node(tri, transform=trans, mixture=water) for tri, trans in zip(triangles, transforms)]

The construction is actually the same.

In [ ]:
end_box_2 = Tree()
end_box_2.nodes[PurePath("endbox")] = pool_node
end_box_2.nodes[PurePath("endbox/box")] = endbox_full_node
end_box_2.exclusive[PurePath("endbox")] = [(PurePath("endbox/box"), endbox_full_node)]
for i, triangle in enumerate(triangle_nodes):
    end_box_2.nodes[PurePath(f"endbox/box/tri_{i}")] = triangle
end_box_2.exclusive[PurePath("endbox/box")] = [(PurePath(f"endbox/box/tri_{i}"), triangle) for i, triangle in enumerate(triangle_nodes)]
end_box_2.nodes[PurePath("endbox/box/waterleg")] = endbox_leg_flow
end_box_2.exclusive[PurePath("endbox/box")].extend([(PurePath("endbox/box/waterleg"), endbox_leg_flow)])
end_box_2.plot(label_offset=0.15)

If we got this right, the slanted planes should be the same as those we calculated before in Option 1:

In [ ]:
corners

In [ ]:
planes = set()
for _, c in end_box_2.named_components():
    surfs = c.geometry.surfaces
    planes |= {s for s in surfs if isinstance(s, Plane) and s.a[0] != 0 and s.a[1] !=0}
list(planes)

Moving on to the main part of the rod.

## The Active Region

The main region is built of 2 side plates which hold 21 plates in position. In standard and type 2 fuel, every other plate slot also holds a burnable absorber Cadmium wire.
We will cover the standard fuel in this example, you can work on doing the other ones on your own as an exercise, if you feel like it. It's a good mental exercise to think what would be different, even if you don't go about writing the model down.

In practice, the swaging process can somewhat reshape the exact geometry of how the wire is placed next to the fuel plate within the side plate. For example, the swaging process very well may squish them together such that no water gap is left, and instead the Aluminium comb-hairs are tilted and driven into the plate and the wire, leaving a larger water gap on the other side.
In our model, we will ignore this swaging process effect, because it is unclear exactly what form it will take. This is also how the problem is specified in the original paper.

Therefore, our side plates form a comb-shape. We can build the side plates either as a complex union geometry, or we can split it into multiple pieces, each of which holds just one plate.
The upside of making it one piece is that it is indeed just one piece in practice. However, that creates a region that is very geometrically complex, which may adversely affect transport solver runtimes. Having multiple, smaller cells may be better, but that also comes at the cost that there are more regions, which has its own costs in some cases.
There are no free lunches, though, if we want our model to be exact.
We could smear things together and approximate the problem, which you may want to do if the computational cost is prohibitive for you, but we are not going to run this model so we can afford to have more pieces with less approximations.
In this guide we will use the multiple-cell model for the side plate.

The shape is still too convoluted, because it is a complex comb shape. The side plate is therefore going to be a connection of boxes, the plates are surrounded by a water film that fits in their slot and between them is water in another box.
The upside is that we get a very simple problem for the transport solver, but we pay the price that if someone wants to edit the material of a side plate region they must edit all the pieces, and that there are multiple water cells to think about when changing temperatures.
Oh well, there are tradeoffs in life. ¯\\_(ツ)_/¯

Another option would be to absorb some of the inactive region of the plates into the sideplate, and ignore if the math says the plate is supposed to have a small water film between it and the sideplate. This will only affect the water and aluminium content negligibly, and make the rods much simpler, but we're going for fidelity in this example, because we want to show off how things could be done.
Doing it this way (which makes things simpler both for the researcher and the computer, at the cost of some fidelity) is left to the reader.

The structure is therefore going to be of the form:

```mermaid
flowchart TB
A[Pool] -->|exclusive| R[Rod Virtual Box]
R -->|inclusive| B[Sideplate Virtual]
R -->|inclusive| WG1EX[Water Gap Ex 1]
R -->|inclusive| C1[Plate Water Film 1]
R -->|inclusive| WG12[Water Gap Plates 1-2]
R -->|inclusive| C2[Plate Water Film 2]
R -->|inclusive| WG23[Water Gap Plates 2-3]
R -->|inclusive| More
R -->|inclusive| C3[Plate Water Film 3]
subgraph SidePlate
B -->|inclusive| Al[Aluminium Block]
B -->|inclusive| TE1[Edge Tooth Left]
B -->|inclusive| S1[Plate Slot 1]
B -->|inclusive| T1[Tooth1]
B -->|inclusive| S2[Plate Slot 2]
B -->|inclusive| T2[Tooth2]
B -->|inclusive| Dots2[...]
B -->|inclusive| TE2[Edge Tooth Right]
S1 -->|inclusive| IA1[Inactive Plate 1]
S1 -->|inclusive| WS1[NoWire Aluminium 1]
S2 -->|inclusive| IA2[Inactive Plate 2]
S2 -->|inclusive| WS2[Wire Slot 2]
WS2 -->|exclusive| W2[Wire 2]
end

subgraph Plate 1
C1 -->|exclusive| CL1[Plate 1 Clad]
CL1 -->|exclusive| M1[Plate 1 Meat]
end
subgraph Plate 2
C2 -->|exclusive| CL2[Plate 2 Clad]
CL2 -->|exclusive| M2[Plate 2 Meat]
end
subgraph Plate 3
C3 -->|exclusive| CL3[Plate 3 Clad]
CL3 -->|exclusive| M3[Plate 3 Meat]
end
subgraph More [More Plates]
Dots[...]
end

```

So we see that our pieces are the fuel plates, which are relatively simple, and the sideplates, which are relatively complicated. Let's start with the easier plates.

### Fuel Plates

To model the plates we need to do material specification before we can build the tree, so we might as well do it now.
A deeper dive on material specification is given in a [different notebook](./Material%20Specification.ipynb), and you can look there if you want to deep dive into how that's done.
We are using tables 4-8 in the specification for material specification.

#### Fuel Materials

In [ ]:
type1_fuel = Mixture.by_weight_fraction(
    {Al:   47.697e-2,
     Si:   3.9322e-2,
     U234: 0.0737e-2,
     U235: 9.4962e-2,
     U236: 0.0997e-2,
     U238: 38.602e-2,
     B:    0.000671e-2, #EBC
    }, 
    density=4.3530, #g/cc
    temperature=20.,
)

type2_fuel = Mixture.by_weight_fraction(
    {Al:   28.281e-2,
     Si:   5.3968e-2,
     U234: 0.1012e-2,
     U235: 13.085e-2,
     U236: 0.1368e-2,
     U238: 52.928e-2,
     B:    0.000548e-2, #EBC
    }, 
    density=5.7187, #g/cc
    temperature=20.,
)

standard_fuel = Mixture.by_weight_fraction(
    {Al:   20.325e-2,
     Si:   5.9969e-2,
     U234: 0.1124e-2,
     U235: 14.588e-2,
     U236: 0.1520e-2,
     U238: 58.766e-2,
     B:    0.000498e-2, #EBC
    }, 
    density=6.4820, #g/cc
    temperature=20.,
)

Let's do some sanity checks. We know the uranium density of each type from table 3 of the specification. We expect to be within significant digits of table 3.

In [ ]:
def u_content(mix):
    return sum(d for k, d in mix.weight_densities().items() 
               if k.Z == U235.Z)

assert math.isclose(u_content(standard_fuel), 4.8, abs_tol=0.05)
assert math.isclose(u_content(type2_fuel), 3.8, abs_tol=0.05)
assert math.isclose(u_content(type1_fuel), 2.1, abs_tol=0.05)

We also expect the enrichment to be 19.8wt%

In [ ]:
def enrich_check(mix, name):
    total_u = u_content(mix)
    _u235 = mix.weight_densities()[U235]
    assert math.isclose(_u235/total_u, 19.8e-2, abs_tol=0.05e-2), f"{name} Enrichment: {_u235/total_u:.1%}, expected 19.8%"

enrich_check(standard_fuel, "Standard Fuel")
enrich_check(type2_fuel, "Type 2 Fuel")
try:
    enrich_check(type1_fuel, "Type 1 Fuel")
except AssertionError as e:
    print("Type 1 Fuel specification has the wrong enrichment, but that's a specification problem!")
    print(e)

Oh well, sometimes your specification isn't consistent. That's life.
We note this in case we can find a bug later, or get some confirmation from the original authors by email that their specification is a bit lax on this point, and move on.

The Cadmium wire (Burnable absorber) is specified with a minimal amount of Cadmium rather than an expected estimate.
This means that we need to make a choice, because we can't do best-estimate on edge-specification directly.

For this guide we'll use the minimal amount of BA, but proper best-estimate specifications may use different values to try to get at the actual estimate.

In [ ]:
cadmium_ba = Mixture.by_weight_fraction(
    composition={Cd: 99.95e-2}, # We have 0.05% at-most unknown stuff.
    density=8.64,
    temperature=20.,
)

#### Plate Specification
The geometrical specification of the fuel is given in Table 3 in the specification on pages 3-4. Standard and type-2 fuels have the same geometry, whereas type-1 fuel has no BA. We'll keep that in mind for when we assemble the rods, but for now we want plate specification, and those are the same in all 3 cases, but there are internal and external plates to handle, that have different geometries.

We actually do have a choice to make now, however. In the plan above we described the plates as a "picture in a frame", with an exclusive relationship between the al6061 clad "frame" and the meat "picture". This saves on tree complexity and cells in an OpenMC model, but the "frames" will have a geometry with many surfaces.
Having 27 sibling boxes instead of a nested relationship may sometimes be preferable, in some use cases. However, it makes the model more complicated. We will go with the simpler, nested behavior described above.
One can convert between the two using tools in this package, such as `coremaker.elements.box.excludeframe_to_framebox`. To create this subgraph directly in a 27-piece tree we would use `coremaker.elements.box.FrameBox`. Instead, we use `coremaker.elements.box.ExcludeFrame`.

The assemblies in the reactor are sometimes rotated. We will use a case where the plates are perpendicular to the x-axis.

The plate thicknesses are given both as nominal values and as actual measurements. Those are within tolerance. One must choose whether they want to model the nominal reactor specification or the as-made numbers.
If you go as-made, you would have to re-allocate the water gaps between
plates to account for the manufacturing discrepancy within tolerance.

We will do the nominal model, not the as-made one, for simplicity.

In [ ]:
MEAT_THICKNESS = 0.61e-1  # mm to cm
CLAD_THICKNESS_IN = 0.37e-1
CLAD_THICKNESS_EX = 0.445e-1
MEAT_WIDTH = 65e-1
PLATE_WIDTH = 75e-1
MEAT_HEIGHT = 615e-1
PLATE_HEIGHT_IN = 655e-1
PLATE_HEIGHT_EX = 825e-1
PLATE_THICK_IN = MEAT_THICKNESS + 2*CLAD_THICKNESS_IN
PLATE_THICK_EX = MEAT_THICKNESS + 2*CLAD_THICKNESS_EX
EX_SHIFT = 148e-1 - (PLATE_HEIGHT_EX - MEAT_HEIGHT)/2
# We don't need a shift for internal plates because 
# 20mm = (655-615)/2 exactly matches a symmetric placement.

In [ ]:
SIDE_THICK = 5e-1
SIDE_WIDTH = 80.5e-1
SIDE_LENGTH = 890e-1
SIDE_SLOT_DEPTH = 2.5e-1
SIDE_SLOT_WIDTH_IN = 1.45e-1
SIDE_SLOT_WIDTH_EX = 1.6e-1
SIDE_BA_GROOVE_DEPTH = 0.6e-1
SIDE_BA_GROOVE_WIDTH = 0.5e-1
PLATE_GAP = 2.45e-1
PLATE_PITCH_IN = PLATE_THICK_IN + PLATE_GAP
TOOTH_WIDTH_IN = PLATE_PITCH_IN - SIDE_SLOT_WIDTH_IN
PLATE_PITCH_EX = PLATE_GAP + PLATE_THICK_IN/2 + PLATE_THICK_EX/2
TOOTH_WIDTH_EX = PLATE_PITCH_EX - SIDE_SLOT_WIDTH_EX/2 - SIDE_SLOT_WIDTH_IN/2
END_WIDTH = 0.5 * (SIDE_WIDTH 
                  - 19*SIDE_SLOT_WIDTH_IN 
                  - 2*SIDE_SLOT_WIDTH_EX
                  - 18*TOOTH_WIDTH_IN
                  - 2*TOOTH_WIDTH_EX
                 )

There is one more decision to make, which is the resolution we want for the meat, for when the material starts its burnup and is no longer homogeneous.

A proper model would at least be split in the height-axis, and probably in the width-axis. This will make the model less readable right now, though, and we don't do any burnup in this example, but if we wanted a burnup-ready model we would use the `picture_resolution` parameter in the `ExcludeFrame` function to ensure our model is split up for burnup at our desired resolution.

In [ ]:
def plate(meat_mix: Mixture, meat_resolution=(math.inf, math.inf, math.inf), *,
          plate_thick, plate_height, side_slot_width, shift):
    exframe = ExcludeFrame(frame_dimensions=(plate_thick, PLATE_WIDTH, plate_height),
                           picture_dimensions=(MEAT_THICKNESS, MEAT_WIDTH, MEAT_HEIGHT),
                           frame_name=PurePath("Cladding"),
                           picture_name=PurePath("Meat"),
                           frame_mixture=al6061clad,
                           picture_mixture=meat_mix,
                           picture_resolution=meat_resolution,
                           picture_translation=(0., 0., shift),
                          )
    water_box: Tree = BoxTree(dimensions=(side_slot_width, SIDE_WIDTH - 2*SIDE_SLOT_DEPTH, SIDE_LENGTH),
                              mixture=water,
                              name=PurePath("PlateWaterFilm"),
                             )
    water_box.graft(exframe, PurePath("PlateWaterFilm"), relation=ChildType.exclusive)
    return water_box


internal_plate = partial(plate, 
                         plate_thick=PLATE_HEIGHT_IN,
                         plate_height=PLATE_HEIGHT_IN,
                         side_slot_width=SIDE_SLOT_WIDTH_IN,
                         shift=0.,
                        )
external_plate = partial(plate,
                         plate_thick=PLATE_HEIGHT_EX,
                         plate_height=PLATE_HEIGHT_EX,
                         side_slot_width=SIDE_SLOT_WIDTH_EX,
                         shift=EX_SHIFT,
                        )


def water_plate_gap(name, gap):
    return BoxTree(dimensions=(gap, SIDE_WIDTH - 2*SIDE_THICK, SIDE_LENGTH),
                   mixture=water,
                   name=PurePath(name),
                  )

In [ ]:
example_plate = internal_plate(standard_fuel)
example_plate_for_burnup = internal_plate(standard_fuel, (math.inf, 2.5, 7.))

Our non-burnup model is very simple to visualize.

In [ ]:
example_plate.plot(size=(10, 2), label_offset=2e-2)

The burnup-ready plate isn't easy to visualize. It has dozens of pieces in its virtual meat, but that's the model you're more likely to use, since you want to capture the burnup gradients over the plate.

Now we need to make the side plate.

#### BA Specification
Again, we go with the nominal values even though the measured values are available. One would choose the proper nominal vs as-made values according to their need.

The BA definitely needs to be split radially and axially, but again we will avoid it for simplicity, and show how that would be done.

In [ ]:
BA_RADIUS = 0.5e-1 / 2
BA_LENGTH = 308e-1
BA_SHIFT = 123e-1 - (MEAT_HEIGHT - BA_LENGTH)/2
BA_SHIFT

In [ ]:
def ba_wire():
    return CylinderTree(radius=BA_RADIUS,
                        length=BA_LENGTH,
                        mixture=cadmium_ba,
                        root_path=PurePath("BAWire"),
                        axis=(0,0,1),
                       )
example_wire = ba_wire()

def ba_wire_burnup():
    return ChunkedCylinderTree(radius=BA_RADIUS,
                               length=BA_LENGTH,
                               mixture=cadmium_ba,
                               root_path=PurePath("BAWire"),
                               axis=(0,0,1),
                               resolution=(0.1e-1, 7.),
                              )

example_burnup_wire = ba_wire_burnup()

The simple wire is very simple, because it is just one cylinder.

In [ ]:
example_wire.plot(size=(2,2), label_offset=3e-2)

The complicated wire isn't as simple, though it is obviously not too bad. Just a little bit complicated.

#### Side Plate Specification
##### Type 1 fuel, with no BA
This is the simpler case, where we don't need to create the grooves for the wires, so let's start with that.

We have two end sections, followed by a bunch of tooth-slot pairs, to create the full comb shape. Let's get on it.

There is a question here, it is slightly unclear if what's conserved in the external plate spacing is the water gap (2.45mm) or the plate center pitch. We decided to keep the water gap rather than the pitch. It makes sense if they use some tool to ensure that the gap is as required and for TH reasons.

This makes the spacing slightly more convoluted, but it's probably the right call here.

In [ ]:
def sideblock(name, thick, width):
    return BoxTree(dimensions=(thick, width, SIDE_LENGTH),
                   mixture=al6061side,
                   name=PurePath(name)
                  )

In [ ]:
sideorder = (["EndTooth", "ExSlot", "ExTooth"] 
             + sum((["InSlot", "InTooth"] for _ in range(18)), []) 
             + ["InSlot", "ExTooth", "ExSlot", "EndTooth"]
            )
sidemapping = {"EndTooth": (END_WIDTH, SIDE_THICK, ),
               "ExSlot": (SIDE_SLOT_WIDTH_EX, SIDE_THICK - SIDE_SLOT_DEPTH),
               "ExTooth": (TOOTH_WIDTH_EX, SIDE_THICK),
               "InSlot": (SIDE_SLOT_WIDTH_IN, SIDE_THICK - SIDE_SLOT_DEPTH),
               "InTooth": (TOOTH_WIDTH_IN, SIDE_THICK),
              }

def sidenum(kind, i):
    return {"EndTooth": "1" if i == 0 else "2",
            "ExSlot": str(i),
            "InSlot": str(i),
            "ExTooth": f"{i}to{i+1}",
            "InTooth": f"{i}to{i+1}",
           }[kind]

def type1fuel_region():
    loc = -SIDE_WIDTH/2
    sideplate_funcs = []
    sideplate_centers = []
    sideplate_names = []
    active_funcs = []
    active_centers = []
    active_names = []
    slotnum = 0
    for kind in sideorder:
        thick, width = sidemapping[kind]
        loc += thick/2
        lcenter = (loc, -SIDE_WIDTH/2 + width/2, 0.)
        rcenter = (loc, SIDE_WIDTH/2 - width/2, 0.)
        func = partial(sideblock, kind, thick, width)
        if kind in {"ExSlot", "InSlot"}:
            slotnum += 1
        ac_func = {"EndTooth": partial(water_plate_gap, "EndGap", thick),
                   "ExTooth": partial(water_plate_gap, "ExGap", thick),
                   "InTooth": partial(water_plate_gap, "InGap", thick),
                   "ExSlot": partial(external_plate, meat_mix=type1_fuel),
                   "InSlot": partial(internal_plate, meat_mix=type1_fuel),
                  }[kind]
        cname = sidenum(kind, slotnum)
        lname = "Left" + cname
        rname = "Right" + cname
        sideplate_funcs += 2*[func]
        sideplate_centers += [Transform(translation=lcenter), Transform(translation=rcenter)]
        sideplate_names += [lname, rname]
        active_funcs.append(ac_func)
        active_centers.append(Transform(translation=(loc, 0., 0.)))
        active_names.append(cname)

    
    sideplate_pieces = zip(sideplate_funcs, sideplate_centers, sideplate_names)
    active_pieces = zip(active_funcs, active_centers, active_names)
    factories = chain(sideplate_pieces, active_pieces)
    return singular_root_construction(
        factories=factories, 
        outer_geometry=BoxGeometry(center=(0,0,0),
                                   dimensions=(SIDE_WIDTH, SIDE_WIDTH, SIDE_LENGTH)),
        root_path=PurePath("PlateRegion"),
        relationship=ChildType.inclusive)

t1fuel_example = type1fuel_region()

All we now have to do is connect the active and end box regions together.

In [ ]:
def type1_fuel_rod():
    root = BoxTree(dimensions=(CELL_SIDE, CELL_SIDE, 1045e-1), mixture=None, name=PurePath("Unitcell"))
    root.graft(type1fuel_region(), PurePath("Unitcell"), ChildType.inclusive)
    root.graft(deepcopy(end_box_2), PurePath("Unitcell"), ChildType.inclusive)
    return root
print(type1_fuel_rod())

And TADA! You have a type 1 fuel rod.
We made this a factory because usually you need factories that churn out multiple rods of the same type.
It is possible to also imbue the created rod with its kind or even its serial number, if you want to keep track of specific rods later in life.
Simply put such an attribute on your Tree, and follow it.

Similarly, we can build the standard and type2 fuels with the cadmium wires.
This simply involves having sometimes 3 regions when sweeping along the rod, because we have the wire region as well.

This is left as an excersie to the reader.